In [ ]:
import pandas as pd
import numpy as np
from numpy.random import default_rng

import os
import re
import itertools
import fflucsim as ff
import pysalvador as sal
import mathieu as mh

from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

from matplotlib import pyplot as plt
from matplotlib import colormaps
from matplotlib import colors as clr
from matplotlib.lines import Line2D
from adjustText import adjust_text
import textalloc as ta

import seaborn as sns

import importlib
import pickle as pkl
from progressbar import ProgressBar

# Import metadata

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/fflucsim/'
fig_path = f'{base_dir}fig/'

In [ ]:
colors = {'mr': '#9cd3cc', 'cr': '#00463b', 'rr_fc': '#00463b', 'rr': '#00463b'}
cmap_greens = clr.LinearSegmentedColormap.from_list('green_mr_rr', ['#9cd3cc', '#00463b'], N=256)

In [ ]:
variable_alias = {'w_mono':r'$\omega^A$',
                 'w_triso':r'$\omega^T$',
                 'mu_A':r'$\mu^A$',
                 'mu_R':r'$\mu^R$',
                 'mu_T':r'$\mu^T$',}

In [ ]:
assays_info = pd.read_csv('assays_info.csv')
assays_info['log10_w_mono'] = assays_info['w_mono'].apply(lambda x: f'{np.log10(x):.1f}')
assays_info['log10_mu_A'] = assays_info['mu_A'].apply(lambda x: f'{np.log10(x):.1f}')
assays_info['log10_mu_R'] = assays_info['mu_R'].apply(lambda x: f'{np.log10(x):.1f}')

populations_info = pd.read_csv('populations_info.csv')
populations_info['log10_w_mono'] = populations_info['w_mono'].apply(lambda x: f'{np.log10(x):.1f}')
populations_info['log10_mu_A'] = populations_info['mu_A'].apply(lambda x: f'{np.log10(x):.1f}')
populations_info['log10_mu_R'] = populations_info['mu_R'].apply(lambda x: f'{np.log10(x):.1f}')

# Chromosome-matched simulations

assays_info_chrom = pd.read_csv('assays_info_chrom.csv')
assays_info_chrom['log10_w_mono'] = assays_info_chrom['w_mono'].apply(lambda x: np.round(np.log10(x), 2))
assays_info_chrom['log10_w_triso'] = assays_info_chrom['w_triso'].apply(lambda x: np.round(np.log10(x), 2))
assays_info_chrom['log10_mu_A'] = assays_info_chrom['mu_A'].apply(lambda x: np.round(np.log10(x), 1))
assays_info_chrom['log10_mu_R'] = assays_info_chrom['mu_R'].apply(lambda x: np.round(np.log10(x), 1))
assays_info_chrom['log10_mu_T'] = assays_info_chrom['mu_T'].apply(lambda x: np.round(np.log10(x), 1))

populations_info_chrom = pd.read_csv('populations_info_chrom.csv')
populations_info_chrom['log10_w_mono'] = populations_info_chrom['w_mono'].apply(lambda x: np.round(np.log10(x), 2))
populations_info_chrom['log10_w_triso'] = populations_info_chrom['w_triso'].apply(lambda x: np.round(np.log10(x), 2))
populations_info_chrom['log10_mu_A'] = populations_info_chrom['mu_A'].apply(lambda x: np.round(np.log10(x), 1))
populations_info_chrom['log10_mu_R'] = populations_info_chrom['mu_R'].apply(lambda x: np.round(np.log10(x), 1))
populations_info_chrom['log10_mu_T'] = populations_info_chrom['mu_T'].apply(lambda x: np.round(np.log10(x), 1))

assays_info_repl = pd.read_csv('assays_info_repl.csv')
assays_info_repl['log10_mu_A'] = assays_info_repl['mu_A'].apply(lambda x: np.round(np.log10(x), 1))
assays_info_repl['log10_mu_R'] = assays_info_repl['mu_R'].apply(lambda x: np.round(np.log10(x), 1))
assays_info_repl['log10_w_mono'] = assays_info_repl['w_mono'].apply(lambda x: np.round(np.log10(x), 2))

populations_info_repl = pd.read_csv('populations_info_repl.csv')
populations_info_repl['log10_mu_A'] = populations_info_repl['mu_A'].apply(lambda x: f'{np.log10(x):.1f}')
populations_info_repl['log10_mu_R'] = populations_info_repl['mu_R'].apply(lambda x: f'{np.log10(x):.1f}')
populations_info_repl['log10_w_mono'] = populations_info_repl['w_mono'].apply(lambda x: f'{np.log10(x):.1f}')

# Import LRH data

In [ ]:
LRH_data = pd.read_csv('../data/final_fluctuation_data_12132025.csv', index_col=0)\
.dropna(axis=1, how='all').dropna(axis=0, how='all').T.astype({'Chrom':int,
                                                              'Fitness':float,
                                                              'VolumeTotal':int,
                                                              'VolumeSelective':int,
                                                              'DilutionSelective':int,
                                                              'CountsSelective':float,
                                                              'VolumeNonselective':int,
                                                              'DilutionNonselective':int,
                                                              'CountsNonselective':float,
                                                              'LOD':float,
                                                              '# cells plated':float,
                                                              'eff':float,
                                                              'Nt':float,
                                                              'CV':float,
                                                              'Fitness':float,
                                                              'Lag':int,
                                                              'Death':int,
                                                              'Residual':int,
                                                              'Inoculum':int,
                                                              'm':float,
                                                               'm_low':float,
                                                               'm_up':float,
                                                               'mu':float,
                                                               'mu_low':float,
                                                               'mu_up':float})
LRH_data['n_reps'] = LRH_data.apply(lambda x: x['CountsSelective'].dropna().shape[0], axis=1)
mono_fitness_chrom = LRH_data.set_index('MutType').loc['mono'].groupby('Chrom').apply(lambda x: x['Fitness'].iloc[0])
LRH_data['fitness'] = mono_fitness_chrom.loc[LRH_data['Chrom']].values

repl_chrom = [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16]
LRH_data['repl'] = (LRH_data['ExptType']=='replating') & \
(LRH_data['FitnessCorr']=='TRUE') & \
(LRH_data['Chrom'].isin(repl_chrom))

In [ ]:
chrom_rates = []
for chrom, df in LRH_data.set_index(['ExptType', 'MutType', 'FitnessCorr']).sort_index().groupby('Chrom'):
    
    mr, mr_low, mr_up, canr_Nt, canr_eff = df.loc[('culture', 'mono', 'TRUE'), ['mu', 'mu_low', 'mu_up', 'Nt', 'eff']]
    
    cr, cr_low, cr_up = df.loc[('culture', 'rev', 'TRUE'), ['mu', 'mu_low', 'mu_up']]
    rr_fc = cr/mr
    
    canr_countsSel = df.loc[('culture', 'rev', 'TRUE'), 'CountsSelective'].dropna().values/canr_eff
    canr_volNonsel, canr_voltot, canr_dilNonsel = df.loc[('culture', 'mono', 'TRUE'), ['VolumeNonselective', 'VolumeTotal', 'DilutionNonselective']]
    canr_effNonsel = canr_volNonsel/(canr_voltot*canr_dilNonsel)
    canr_countsNonsel = df.loc[('culture', 'rev', 'TRUE'), 'CountsNonselective'].dropna().values/canr_effNonsel
    canr_rev_ratio = canr_countsSel/canr_countsNonsel
    canr_mean_rev_ratio = canr_rev_ratio.mean()
    
    if chrom in repl_chrom:
        rr, rr_low, rr_up, repl_Nt, repl_eff = df.loc[('replating', 'rev', 'TRUE'), ['mu', 'mu_low', 'mu_up', 'Nt', 'eff']]
        
        repl_countsSel = df.loc[('replating', 'rev', 'TRUE'), 'CountsSelective'].dropna().values
        repl_countsNonsel = df.loc[('replating', 'rev', 'TRUE'), 'CountsNonselective'].dropna().values
        repl_rev_ratio = repl_countsSel/repl_countsNonsel
        repl_mean_rev_ratio = repl_rev_ratio.mean()
    else:
        rr, rr_low, rr_up, repl_Nt, repl_eff, repl_countsSel, repl_countsNonsel, repl_rev_ratio, repl_mean_rev_ratio = np.repeat(np.nan, 9)
        
    chrom_rates.append([chrom, mono_fitness_chrom.loc[chrom],
                        mr, mr_low, mr_up, canr_Nt, canr_eff,
                        cr, cr_low, cr_up,
                        canr_effNonsel, canr_rev_ratio, canr_mean_rev_ratio, canr_countsSel, canr_countsNonsel,
                        rr, rr_low, rr_up, repl_Nt, repl_eff,
                        rr_fc, repl_rev_ratio, repl_mean_rev_ratio, repl_countsSel, repl_countsNonsel])

chrom_rates = pd.DataFrame(chrom_rates, columns=['chrom', 'fitness',
                                                 'mr', 'mr_low', 'mr_up', 'canr_Nt', 'canr_eff',
                                                 'cr', 'cr_low', 'cr_up',
                                                 'canr_effNonsel', 'canr_rev_ratio', 'canr_mean_rev_ratio', 'canr_countsSel', 'canr_countsNonsel',
                                                 'rr', 'rr_low', 'rr_up', 'repl_Nt', 'repl_eff',
                                                 'rr_fc', 'repl_rev_ratio', 'repl_mean_rev_ratio', 'repl_countsSel', 'repl_countsNonsel'])
chrom_rates = chrom_rates.set_index('chrom')

# Validation of fflucsim

In [ ]:
PopReport = []
idx_bar = 0
with ProgressBar(max_value=assays_info.shape[0]) as bar:
  
    for a in assays_info['assay']:
        pop_rep = pd.read_csv(f'{base_dir}populations/{a}.PopReport.csv')
        pop_rep['select'] = False
        PopReport.append(pop_rep)

        pop_rep_sel = pd.read_csv(f'{base_dir}populations/{a}.PopReportSelect.csv')
        pop_rep_sel['select'] = True
        PopReport.append(pop_rep_sel)

        idx_bar += 1
        bar.update(idx_bar)

PopReport = pd.concat(PopReport).reset_index(drop=True)

In [ ]:
FA_Store = {}
FA_Results = []

for a in assays_info['assay'].values:

    fn = f'fa.{a}.pkl'
    fp = f'../data/pickle/{fn}'
    if fn in os.listdir('../data/pickle/'):
        with open(fp, 'rb') as fi:
            FA = pkl.load(fi)
            
            for r in FA.Results:
                r['assay'] = a
                FA_Results.append(pd.Series(r))
            FA_Store[a] = FA

FA_Results = pd.DataFrame(FA_Results).merge(assays_info, on='assay')

## Fig S4

In [ ]:
fig = plt.figure(figsize=[12,8])
gs = plt.GridSpec(ncols=1, nrows=5, height_ratios=[4,1,0.5,4,1], hspace=0.1,
                  left=0.08, right=0.9, bottom=0.05, top=0.95)

model_color = {'LD':'w', 'MK':'k'}
model_offset = {'LD':-0.2, 'MK':0.2}

## Monosome rate

A = assays_info.sort_values(by=['mu_A', 'w_mono', 'mu_R'])['assay']
X = dict(zip(A, np.arange(A.shape[0])))

ax = fig.add_subplot(gs[0])
for mu_A, df in FA_Results.loc[(FA_Results['assay'].isin(A)) & (FA_Results['mutant']=='monosome') & (FA_Results['upper_bound']==False)].groupby('mu_A'):

    x_span = df['assay'].apply(lambda x: X[x])
    x_span = (x_span.min(), x_span.max())
    ax.plot(x_span, [mu_A, mu_A], c='k', lw=0.5, zorder=0)
    
    for i, (a, model, mu, mu_ci) in df[['assay', 'model', 'mu', 'mu_ci']].iterrows():
        if not pd.isna(mu):
            x = X[a]+model_offset[model]
            ax.plot([x,x], mu_ci, color='k', lw=0.5, zorder=1)
            ax.scatter([x], [mu], s=10, lw=0.5, ec='k', fc=model_color[model], zorder=1)

ax.set_xlim(-3, 248)
ax.set_xticks([])
ax.set_xlabel('')
ax.set_yscale('log')
ax.set_ylabel(r'$\mu^A$')

legend_emlms = [Line2D([0], [0], c='w', marker='o', ms=5, mfc=c, mec='k', mew=0.5, label=m) for (m,c) in 
               zip(['Lea-Coulson', 'Mandelbrot-Koch'], ['w', 'k'])]
ax.legend(handles=legend_emlms, title='model', loc=2, bbox_to_anchor=[0.88, 0.5], frameon=False, fontsize=14, title_fontsize=14)

ax = fig.add_subplot(gs[1])
dat = assays_info.set_index('assay').loc[A, ['mu_A', 'w_mono', 'mu_R']].map(np.log10).T

ax.imshow(dat, aspect='auto')
ax.set_xticks([])
ax.set_xlim(-3, 248)
ax.set_yticks([0,1,2])
ax.set_yticklabels([variable_alias[v] for v in dat.index])
for sp in ['left','bottom']:
    ax.spines[sp].set_visible(False)


## Reversion rate

A = assays_info.sort_values(by=['mu_R', 'w_mono', 'mu_A'])['assay']
X = dict(zip(A, np.arange(A.shape[0])))

ax = fig.add_subplot(gs[3])
for (mu_R, w_mono), df in FA_Results.loc[(FA_Results['assay'].isin(A)) & (FA_Results['mutant']=='revert') & (FA_Results['upper_bound']==False)].groupby(['mu_R','w_mono']):


    x_span = df['assay'].apply(lambda x: X[x])
    ax.plot(x_span, df['mu_R']*df['mu_A'], c='k', lw=0.5, zorder=0)
    
    for i, (a, model, mu, mu_ci) in df[['assay', 'model', 'mu', 'mu_ci']].iterrows():
        if not pd.isna(mu):
            x = X[a]+model_offset[model]
            ax.plot([x,x], mu_ci, color='k', lw=0.5, zorder=1)
            ax.scatter([x], [mu], s=10, lw=0.5, ec='k', fc=model_color[model], zorder=1)

ax.set_xlim(-3, 248)
ax.set_xticks([])
ax.set_xlabel('')
ax.set_yscale('log')
ax.set_ylabel(r'$\mu^A\times\mu^R$')

ax = fig.add_subplot(gs[4])
dat = assays_info.set_index('assay').loc[A, ['mu_R', 'w_mono', 'mu_A']].T.map(np.log10)

hm = ax.imshow(dat, aspect='auto')
ax.set_xticks([])
ax.set_xlim(-3, 248)
ax.set_yticks([0,1,2])
ax.set_yticklabels([variable_alias[v] for v in dat.index])
for sp in ['left','bottom']:
    ax.spines[sp].set_visible(False)

cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.3])
cb = fig.colorbar(hm, cbar_ax, drawedges=False)
cb.outline.set_visible(False)
cbar_ax.set_yticks(np.linspace(-7, 0, 8), labels=['1', '10$^{-1}$', '10$^{-2}$', '10$^{-3}$', '10$^{-4}$', '10$^{-5}$', '10$^{-6}$', '10$^{-7}$'][::-1])

fig.text(0.03, 0.96, 'A', fontweight='bold', size=24)
fig.text(0.03, 0.47, 'B', fontweight='bold', size=24)

#for ext in ['png', 'svg']:
#    plt.savefig(f'{fig_path}subm1/FigS4.{ext}', dpi=300)

#plt.show()
plt.close()

## Import population reports

In [ ]:
PopReport = []
idx_bar = 0
with ProgressBar(max_value=assays_info.shape[0]) as bar:
    for a, df in populations_info.groupby('assay'):
        pop_list = df['pop_id'].values
        lofn = [f'../data/pickle/pop_wt.{p}.pkl' for p in pop_list]
        Populations = ff.load_populations(lofn)
    
        poprep = pd.concat([pd.Series(pop.Report) for pop in Populations], axis=1).T
        poprep['assay'] = a
        poprep['pop'] = pop_list
        poprep['chrom'] = chrom

        PopReport.append(poprep)

        idx_bar += 1
        bar.update(idx_bar)

PopReport = pd.concat(PopReport).reset_index(drop=True)

In [ ]:
PopReport['revert_ratio'] = PopReport['n_revert']/PopReport['n_total']
PopReport['monosome_ratio'] = PopReport['n_monosome']/PopReport['n_total']

PopReport['log10_monosome_rate'] = PopReport['monosome_rate'].apply(lambda x: np.round(np.log10(x), 1))
PopReport['log10_revert_rate'] = PopReport['revert_rate'].apply(lambda x: np.round(np.log10(x), 1))
PopReport['log10_monosome_fitness'] = PopReport['monosome_fitness'].apply(lambda x: np.round(np.log10(x), 2))

# Chromosome simulations

In [ ]:
chrom_log_mu_A_order = np.sort(assays_info_chrom['log10_mu_A'].unique())
chrom_log_mu_R_order = np.sort(assays_info_chrom['log10_mu_R'].unique())
chrom_log_mu_T_order = np.sort(assays_info_chrom['log10_mu_T'].unique())
chrom_log_w_mono_order = np.sort(assays_info_chrom['log10_w_mono'].unique())
chrom_log_w_triso_order = np.sort(assays_info_chrom['log10_w_triso'].unique())

In [ ]:
PopReportChrom = []
idx_bar = 0
with ProgressBar(max_value=assays_info_chrom.shape[0]) as bar:
  
    for a in assays_info_chrom['assay']:
        pop_rep = pd.read_csv(f'{base_dir}populations_chrom/{a}.PopReport.csv')
        pop_rep['select'] = False
        PopReportChrom.append(pop_rep)

        pop_rep_sel = pd.read_csv(f'{base_dir}populations_chrom/{a}.PopReportSelect.csv')
        pop_rep_sel['select'] = True
        PopReportChrom.append(pop_rep_sel)

        idx_bar += 1
        bar.update(idx_bar)

PopReportChrom = pd.concat(PopReportChrom).reset_index(drop=True)

In [ ]:
PopReportChrom['target_div'] = chrom_rates['canr_Nt'].loc[PopReportChrom['chrom']].values
PopReportChrom['monosome_ratio'] = PopReportChrom['n_monosome']/PopReportChrom['target_div']
PopReportChrom['revert_ratio'] = PopReportChrom['n_revert']/PopReportChrom['target_div']
PopReportChrom['revert2_ratio'] = PopReportChrom['n_revert2']/PopReportChrom['target_div']
PopReportChrom['revert_total_ratio'] = PopReportChrom['revert_ratio'] + PopReportChrom['revert2_ratio']

PopReportChrom['log10_w_mono'] = PopReportChrom['w_mono'].apply(lambda x: np.round(np.log10(x), 2))
PopReportChrom['log10_w_triso'] = PopReportChrom['w_triso'].apply(lambda x: np.round(np.log10(x), 2))
PopReportChrom['log10_mu_A'] = PopReportChrom['mu_A'].apply(lambda x: np.round(np.log10(x), 1))
PopReportChrom['log10_mu_R'] = PopReportChrom['mu_R'].apply(lambda x: np.round(np.log10(x), 1))
PopReportChrom['log10_mu_T'] = PopReportChrom['mu_T'].apply(lambda x: np.round(np.log10(x), 1))

## Fig 4B

In [ ]:
fig = plt.figure(figsize=[12,7.5])
gs = plt.GridSpec(ncols=5, nrows=3, hspace=0.55, wspace=0.55,
                 left=0.08, right=0.95, bottom=0.12, top=0.95)

chrom_order = chrom_rates.loc[~chrom_rates['rr'].isna()].index
chrom_ax = dict(zip(chrom_order, list(itertools.product(range(3), range(5)))))

for chrom, df in PopReportChrom.loc[PopReportChrom['select']==True].groupby('chrom'):

    #if chrom == 1:
    if True:
        ax = fig.add_subplot(gs[chrom_ax[chrom]])

        empirical_ratio, Nt = chrom_rates.loc[chrom, ['canr_mean_rev_ratio', 'canr_Nt']]
        lower_bound_ratio = 1/(Nt*50)
        
        dat = df.pivot_table(index='log10_w_triso', columns='log10_mu_T', values='revert_total_ratio', aggfunc='mean')
        
        lower_bounds = np.zeros(dat.shape)
        lower_bounds[dat==0] = 1
        dat[dat==0] = lower_bound_ratio
        dat = np.log10(empirical_ratio/dat)
    
        interpolation = 'none'
        cmap = 'BrBG'
        
        hm_log = ax.imshow(dat.loc[chrom_log_w_triso_order, chrom_log_mu_T_order],
                           vmin=-2, vmax=2, aspect='auto', interpolation=interpolation, cmap=cmap)
        where_lower_bounds = np.where(lower_bounds)
        ax.scatter(where_lower_bounds[1], where_lower_bounds[0], c='k', s=20)
        ax.set_title(f'Chr{chrom}')

        ax.set_yticks(range(dat.shape[0]), labels=chrom_log_w_triso_order)
        ax.set_xticks(range(dat.shape[1]), labels=chrom_log_mu_T_order, rotation=90)
        
        ax.invert_yaxis()

        ax.tick_params(axis='both', color='red', labelcolor='red')
        ax.axis('off')

        ax = fig.add_subplot(gs[chrom_ax[chrom]])
        ax.set_facecolor((1,1,1,0))
        ax.set_xscale('log')
        ax.set_xticks(np.logspace(-4, 0, 3))
        ax.set_xlim(10**(-4-2/9), 10**(2/9))
        
        ax.set_yscale('log')
        ax.set_ylim(10**(-2-1/9), 10**(1/9))
        
cax_log = fig.add_axes([0.85, 0.05, 0.015, 0.2])
cb_log = fig.colorbar(hm_log, cax=cax_log, drawedges=False)
cbar_log_yticks = np.linspace(-2, 2, 5)
cax_log.set_yticks(cbar_log_yticks)
cax_log.set_yticklabels([f'{10**y:.2f}' for y in cbar_log_yticks], size=14)
cax_log.text(0.5, 1.1, r'$\mathsf{\frac{empirical.freq.R}{simulated.freq.R}}$', 
             size=18, ha='center', va='bottom', transform=cax_log.transAxes)
#cb_log.outline.set_visible(False)

fig.text(0.02, 0.5, r'$\omega^T$', ha='center', va='center', size=16, rotation=90)
fig.text(0.5, 0.03, r'$\mu^T$', ha='center', va='center', size=16)

#for ext in ['png','svg']:
#    plt.savefig(f'{fig_path}subm1/Fig4B.{ext}', dpi=300)
#plt.show()
plt.close()

# Monosome replating simulations

In [ ]:
repl_log_revert_rate_order = np.sort(assays_info_repl['log10_mu_R'].unique())
repl_log_mono_fitness_order = np.sort(assays_info_repl['log10_w_mono'].unique())

## Export the data for replating fit with varying w_mono

In [ ]:
C = ['Fitness',
     'Death',
     'Residual',
     'Inoculum',
     'VolumeTotal',
     'VolumeSelective',
     'DilutionSelective',
     'CountsSelective',
     'VolumeNonselective',
     'DilutionNonselective',
     'CountsNonselective']
dat = LRH_data.loc[(LRH_data['ExptType']=='replating'), C].copy()
assay_order = dat.index
dat['Name'] = [f'{a}_west' for a in assay_order]
export = [dat]
for w_mono in assays_info_repl['w_mono'].unique():
    new_dat = dat.copy()
    new_dat['Fitness'] = 1/w_mono
    new_dat['Name'] = [f'{a}_w{np.round(np.log10(w_mono), 2)}' for a in assay_order]
    export.append(new_dat)

export = pd.concat(export).set_index('Name').T

#export.to_csv(f'{base_dir}data/replating_data_varying_w.csv')

In [ ]:
PopReportRepl = []
idx_bar = 0
with ProgressBar(max_value=assays_info_repl.shape[0]) as bar:
  
    for a in assays_info_repl['assay']:
        pop_rep = pd.read_csv(f'{base_dir}populations_repl/{a}.PopReport.csv')
        pop_rep['select'] = False
        PopReportRepl.append(pop_rep)

        pop_rep_sel = pd.read_csv(f'{base_dir}populations_repl/{a}.PopReportSelect.csv')
        pop_rep_sel['select'] = True
        PopReportRepl.append(pop_rep_sel)

        idx_bar += 1
        bar.update(idx_bar)

PopReportRepl = pd.concat(PopReportRepl).reset_index(drop=True)

In [ ]:
PopReportRepl['target_div'] = chrom_rates['repl_Nt'].loc[PopReportRepl['chrom']].values
PopReportRepl['monosome_ratio'] = PopReportRepl['n_monosome']/PopReportRepl['target_div']
PopReportRepl['revert_ratio'] = PopReportRepl['n_revert']/PopReportRepl['target_div']
PopReportRepl['revert2_ratio'] = PopReportRepl['n_revert2']/PopReportRepl['target_div']
PopReportRepl['revert_total_ratio'] = PopReportRepl['revert_ratio'] + PopReportRepl['revert2_ratio']

PopReportRepl['log10_w_mono'] = PopReportRepl['w_mono'].apply(lambda x: np.round(np.log10(x), 2))
PopReportRepl['log10_w_triso'] = PopReportRepl['w_triso'].apply(lambda x: np.round(np.log10(x), 2))
PopReportRepl['log10_mu_A'] = PopReportRepl['mu_A'].apply(lambda x: np.round(np.log10(x), 1))
PopReportRepl['log10_mu_R'] = PopReportRepl['mu_R'].apply(lambda x: np.round(np.log10(x), 1))
PopReportRepl['log10_mu_T'] = PopReportRepl['mu_T'].apply(lambda x: np.round(np.log10(x), 1))

## Fig 3C

In [ ]:
fig, ax = plt.subplots(figsize=[5,5])

def log_R_cmap(x):
    return 1-np.log10(x)*(-1)/4

X = [9e-7, 2e-5]
ax.plot(X, X, ls='--', c='grey', zorder=-1, clip_on=True)

X = []
Y = []
labels = []
dX = []
dY = []
for chrom, row in chrom_rates.loc[repl_chrom].iterrows():
    x, y = row[['mr', 'rr']]
    dx = row[['mr_low', 'mr_up']]
    dy = row[['rr_low', 'rr_up']]
    mrr = row['repl_mean_rev_ratio']
    mrr_cmap = log_R_cmap(mrr)
    
    ax.scatter(x, y, s=81, 
               #facecolor=colormaps.get_cmap('viridis')(mrr), 
               facecolor=cmap_greens(mrr_cmap),
               edgecolors='k', label=chrom)
    ax.plot(np.repeat(row['mr'], 2), dy, zorder=0, c='k', lw=1.5)
    ax.plot(dx, np.repeat(row['rr'], 2),  zorder=0, c='k', lw=1.5)
    
    X.append(x)
    Y.append(y)
    dX.append(dx)
    dY.append(dy)
    labels.append(chrom)

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel(r'$\mu^A$')
ax.set_ylabel(r'$\mu^{R}$')

ta.allocate(ax, X, Y, labels,
            x_scatter=X, y_scatter=Y, 
           x_lines=dX, y_lines=dY,
           min_distance=0.05, margin=0.025,
           linecolor='k', linewidth=0.5, textsize=12)

legend_elms = [Line2D([0], [0], c='w', marker='o', ms=9, 
                      #mfc=colormaps.get_cmap('viridis')(i), 
                      mfc=cmap_greens(log_R_cmap(i)), 
                      mec='k', mew=1, label=f'{i*100:}%') for i in np.logspace(-4, 0, 5)]
ax.legend(handles=legend_elms, loc=4, bbox_to_anchor=[1,0], frameon=False, title='freq. R', fontsize=12, title_fontsize=12)

plt.tight_layout()
#for ext in ['svg', 'png']:
#    plt.savefig(f'{fig_path}subm1/Fig3C.{ext}', dpi=300)
#plt.show()
plt.close()

In [ ]:
# mlemur output for range of w_mono values

replating_data = pd.read_excel('../data/replating_fit_varying_w.xlsx', index_col=0).T.astype({'eff':float,
                                                                                              'Nt':float,
                                                                                              'Fitness':float,
                                                                                              'm':float,
                                                                                               'm_low':float,
                                                                                               'm_up':float,
                                                                                               'mu':float,
                                                                                               'mu_low':float,
                                                                                               'mu_up':float})
replating_data['Chrom'] = [int(i.split('_')[1]) for i in replating_data.index]
replating_data['fitness'] = mono_fitness_chrom.loc[replating_data['Chrom']].values
replating_data = replating_data.iloc[14:].reset_index(names='name').set_index('Chrom')

## Fig 3D & S5B

In [ ]:
repl_chrom_order = chrom_rates.loc[~chrom_rates['rr'].isna()].index
chrom_ax = dict(zip(repl_chrom_order, list(itertools.product(range(3), range(5)))))


for scale, fn in zip(['lin', 'log'], ['Fig3D', 'FigS5B']):

    fig = plt.figure(figsize=[12, 7.5])
    gs = plt.GridSpec(ncols=5, nrows=3, hspace=0.55, wspace=0.55,
                 left=0.08, right=0.95, bottom=0.12, top=0.95)

    
    for chrom, df in PopReportRepl.loc[PopReportRepl['select']==True].groupby('chrom'):
        
        w, rr, rr_low, rr_up, repl_mean_rev_ratio, repl_Nt = chrom_rates.loc[chrom, ['fitness', 'rr', 'rr_low', 'rr_up', 'repl_mean_rev_ratio', 'repl_Nt']]

        ax_idx = chrom_ax[chrom]
        ax = fig.add_subplot(gs[ax_idx])
    
        lower_bound_ratio = 1/(repl_Nt*50)
        
        dat = df.pivot_table(index='log10_w_mono', columns='log10_mu_R', values='revert_ratio', aggfunc='mean')
        dat[dat==0] = lower_bound_ratio
        
        interpolation = 'none'
        
        if scale == 'log':
            cmap = cmap_greens
            cmap_value = 1-np.log10(repl_mean_rev_ratio)/-5
            ec = 'w'
            
            dat = dat.map(np.log10)
            hm = ax.imshow(dat.loc[repl_log_mono_fitness_order, repl_log_revert_rate_order],
                           vmin=-5, vmax=0, aspect='auto', interpolation=interpolation, cmap=cmap)
        else:
            cmap = colormaps.get_cmap('BrBG')
            cmap_value = repl_mean_rev_ratio
            ec = 'k'
            
            hm = ax.imshow(dat.loc[repl_log_mono_fitness_order, repl_log_revert_rate_order],
                           vmin=0, vmax=1, aspect='auto', interpolation=interpolation, cmap=cmap)
        
        ax.set_yticks(range(dat.shape[0]), labels=repl_log_mono_fitness_order)
        ax.set_xticks(range(dat.shape[1]), labels=repl_log_revert_rate_order, rotation=90)
        ax.tick_params(axis='both', color='red', labelcolor='red')
        ax.axis('off')
        ax.invert_yaxis()
    
        ax = fig.add_subplot(gs[ax_idx])
        ax.set_facecolor((1,1,1,0))
        
        repl_sub = replating_data.loc[chrom]
        
        fc = cmap(cmap_value)
        
        ax.scatter(rr, w, color=fc, ec=ec, linewidth=0.5, s=120, zorder=3)
        ax.plot([rr_low, rr_up], [w,w], color=ec, zorder=2)
        
        ax.plot(repl_sub['mu'], repl_sub['Fitness']**-1, color=ec, zorder=1)
        ax.plot(repl_sub['mu_low'], repl_sub['Fitness']**-1, ls=':', color=ec, zorder=1)
        ax.plot(repl_sub['mu_up'], repl_sub['Fitness']**-1, ls=':', color=ec, zorder=1)
        
        ax.set_xscale('log')
        ax.set_xticks(np.logspace(-7, -1, 4))
        ax.set_xlim(10**(-7-3/9), 10**(-1+3/9))
        
        ax.set_yscale('log')
        ax.set_ylim(10**(-2-1/9), 10**(1/9))
    
        ax.set_title(f'Chr{chrom}')

    fig.text(0.02, 0.5, r'$\omega^A$', ha='center', va='center', size=16, rotation=90)
    fig.text(0.55, 0.03, r'$\mu^R$', ha='center', va='center', size=16)

    cax = fig.add_axes([0.85, 0.08, 0.015, 0.2])
    cb = fig.colorbar(hm, cax=cax, drawedges=False)
    
    if scale == 'log':

        cbar_ticks = np.linspace(-5, 0, 6)
        cax.set_yticks(cbar_ticks)
        cax.set_yticklabels([f'{10**(y+2)}' for y in cbar_ticks], size=14)
        
    else:

        cbar_ticks = np.linspace(0, 1, 5)
        cax.set_yticks(cbar_ticks)
        cax.set_yticklabels([f'{y*100:.0f}' for y in cbar_ticks], size=14)
    
    cax.text(0.5, 1.1, 'freq. R (%)', ha='center', va='bottom', size=14, transform=cax.transAxes)
    
    #for ext in ['png', 'svg']:
    #    plt.savefig(f'{fig_path}subm1/{fn}.{ext}', dpi=300)
    
    #plt.show()
    plt.close()

## Fig S5A

In [ ]:
fig = plt.figure(figsize=[12,8])

gs = plt.GridSpec(ncols=5, nrows=3, wspace=0.45, hspace=0.55, 
                  right=0.94, top=0.94, left=0.08, bottom=0.1)

repl_chrom_order = chrom_rates.loc[~chrom_rates['rr'].isna()].index
chrom_ax = dict(zip(repl_chrom_order, list(itertools.product(range(3), range(5)))))

for chrom in repl_chrom:

    rev_ratio = chrom_rates.loc[chrom, 'repl_rev_ratio']
    
    ax = fig.add_subplot(gs[chrom_ax[chrom]])

    ax.plot(np.sort(rev_ratio), np.linspace(0, 100, rev_ratio.shape[0]), c='k', lw=2, zorder=3)

    assay_closest = (assays_info_repl.set_index(['chrom','assay']).loc[chrom, ['log10_w_mono','log10_mu_R']]-\
                     chrom_rates.loc[chrom, ['fitness', 'rr']].apply(lambda x: np.log10(x)).values).map(np.abs)\
        .sort_values(by=['log10_w_mono','log10_mu_R']).index[0]
   
    for a, df in PopReportRepl.loc[(PopReportRepl['chrom']==chrom) & (PopReportRepl['select']==True)].groupby('assay'):

        d = df['revert_ratio'].sort_values()
        
        if a == assay_closest:

            #test = stats.ks_2samp(rev_ratio, d)
            test = stats.mannwhitneyu(rev_ratio, d)
            closest_pval = test.pvalue
            print(chrom, a)
            c = 'blue'
            z = 2
            lw = 1
            alpha=1

            ax.plot(d, np.linspace(0, 100, d.shape[0]), c=c, lw=lw, alpha=alpha, zorder=z)
            
    
    ax.set_title(f'Chr{chrom}',)
    ax.text(0.5, 0.5, 
            'Mann-Whitney U test\n'+fr'$p=${mh.plot_pval_text(closest_pval)}',
            size=10, c='b', va='center', ha='center', transform=ax.transAxes)
    
    ax.set_xlim(-0.03, 1.03)
    ax.set_xticks(np.linspace(0,1,5))
    ax.set_xticklabels(np.int16(np.linspace(0,100,5)))
    
fig.text(0.02, 0.5, 'cumul. % cultures', ha='center', va='center', size=16, rotation=90)
fig.text(0.5, 0.03, 'freq. R (%)', ha='center', va='center', size=16)

#for ext in ['png', 'svg']:
#    plt.savefig(f'{fig_path}subm1/FigS5A.{ext}', dpi=300)

#plt.show()
plt.close()

## Fig 3E

In [ ]:
repl_chrom_order = chrom_rates.loc[~chrom_rates['rr'].isna()].index
X = np.arange(14)

boxes_emp = []
boxes_sim = []

for chrom in repl_chrom_order:

    rev_ratio = chrom_rates.loc[chrom, 'repl_rev_ratio']*100
    boxes_emp.append(rev_ratio)
    
    assay_closest = (assays_info_repl.set_index(['chrom','assay']).loc[chrom, ['log10_w_mono','log10_mu_R']]-\
                     chrom_rates.loc[chrom, ['fitness', 'rr']].apply(lambda x: np.log10(x)).values).map(np.abs)\
        .sort_values(by=['log10_w_mono','log10_mu_R']).index[0]
   
    df = PopReportRepl.loc[(PopReportRepl['chrom']==chrom) &
        (PopReportRepl['select']==True) &
        (PopReportRepl['assay']==assay_closest)]
    d = df['revert_ratio'].sort_values()*100
    boxes_sim.append(d)

fig, ax = plt.subplots(figsize=[12,3])
ax.boxplot(boxes_sim, positions=X-0.18, widths=0.3, patch_artist=True,
           medianprops=dict(color='k'), boxprops=dict(facecolor='w'))
ax.boxplot(boxes_emp, positions=X+0.18, widths=0.3, patch_artist=True,
          medianprops=dict(color='k'), boxprops=dict(facecolor='0.5'))

ax.set_ylabel('freq. R (%)')
ax.set_xticks(X)
ax.set_xticklabels(repl_chrom_order)

legend_elms = [Line2D([0], [0], c='w', mfc=fc, mec='k', marker='s', label=l) for (fc, l) in zip(['w','0.5'], ['simulated', 'empirical'])]
ax.legend(handles=legend_elms, loc=3, bbox_to_anchor=[0.9, 0.8])

plt.tight_layout()
#for ext in ['png', 'svg']:
#    plt.savefig(f'{fig_path}subm1/Fig3E.{ext}', dpi=300)

#plt.show()
plt.close()

# Generate metadata sheets of FA params
## Validation dataset

In [ ]:
assays_info = []
populations_info = []

monosome_fitness = np.logspace(0, -2, 5)
monosome_rates = np.logspace(-7, -4, 7)
reversion_rates = np.logspace(-7, -1, 7)

n_reps = 50
assay_idx = 0
pop_idx = 0

for (w_mono, mu_A, mu_R) in itertools.product(monosome_fitness, monosome_rates, reversion_rates):
    assay_id = f'a{assay_idx}'
    assay_idx += 1
    target_div = ff.target_div_monosome_rate(mu_A, max_div=24)
    assays_info.append([assay_id, n_reps, w_mono, mu_A, mu_R, target_div])
    
    for n in range(n_reps):
        populations_info.append([assay_id, w_mono, mu_A, mu_R, target_div, f'p{pop_idx}'])
        pop_idx += 1

assays_info = pd.DataFrame(assays_info, columns=['assay', 'n_rep', 'w_mono', 'mu_A', 'mu_R', 'target_div'])
populations_info = pd.DataFrame(populations_info, columns=['assay', 'w_mono', 'mu_A', 'mu_R', 'target_div', 'pop_id'])

#assays_info.to_csv('assays_info.csv', index=False)
#populations_info.to_csv('populations_info.csv', index=False)

## Chromosome population-matched

In [ ]:
assays_info_chrom = []
populations_info_chrom = []

w_triso_chrom = np.logspace(0, -1, 10)
mu_T_chrom = np.logspace(-4, 0, 10)

n_reps = 50
assay_idx = 0
pop_idx = 0

for chrom, (mu_A, mu_R, w_mono, target_div) in chrom_rates.loc[~chrom_rates['rr'].isna(), ['mr', 'rr', 'fitness', 'canr_Nt']].iterrows():

    # mu_A has to be doubled, because empirical value is only half of the events.
    mu_A *= 2
    
    for (w_triso, mu_T) in itertools.product(w_triso_chrom, mu_T_chrom):
        assay_id = f'a{assay_idx}'
        assay_idx += 1
        
        assays_info_chrom.append([assay_id, chrom, n_reps, w_mono, w_triso, mu_A, mu_R, mu_T, target_div])
        
        for n in range(n_reps):
            populations_info_chrom.append([assay_id, chrom, w_mono, w_triso, mu_A, mu_R, mu_T, target_div, f'p{pop_idx}'])
            pop_idx += 1

assays_info_chrom = pd.DataFrame(assays_info_chrom, columns=['assay', 'chrom', 'n_rep', 'w_mono', 'w_triso', 'mu_A', 'mu_R', 'mu_T', 'target_div'])
populations_info_chrom = pd.DataFrame(populations_info_chrom, columns=['assay', 'chrom', 'w_mono', 'w_triso', 'mu_A', 'mu_R', 'mu_T', 'target_div', 'pop_id'])

#assays_info_chrom.to_csv('assays_info_chrom.csv', index=False)
#populations_info_chrom.to_csv('populations_info_chrom.csv', index=False)

## Chromosome population-matched - Replating

In [ ]:
assays_info_repl = []
populations_info_repl = []

w_mono_repl = np.logspace(0, -2, 10)
mu_R_repl = np.logspace(-7, -1, 10)

n_reps = 50
assay_idx = 0
pop_idx = 0

#for repl, target_div in LRH_data.loc[(LRH_data['FitnessCorr']=='TRUE') & (LRH_data['ExptType']=='replating'), 'Nt'].items():
for chrom, (mu_A, mu_R, w_mono, target_div) in chrom_rates.loc[~chrom_rates['rr'].isna(), ['mr', 'rr', 'fitness', 'repl_Nt']].iterrows():

    mu_A *= 2
    
    for (mu_R, w_mono) in itertools.product(mu_R_repl, w_mono_repl):
        assay_id = f'a{assay_idx}'
        assay_idx += 1
        assays_info_repl.append([assay_id, chrom, n_reps, w_mono, 1, mu_A, mu_R, 0, target_div])
        
        for n in range(n_reps):
            populations_info_repl.append([assay_id, chrom, w_mono, 1, mu_A, mu_R, 0, target_div, f'p{pop_idx}'])
            pop_idx += 1

assays_info_repl = pd.DataFrame(assays_info_repl, columns=['assay', 'chrom', 'n_rep', 'w_mono', 'w_triso', 'mu_A', 'mu_R', 'mu_T', 'target_div'])
populations_info_repl = pd.DataFrame(populations_info_repl, columns=['assay', 'chrom', 'w_mono', 'w_triso', 'mu_A', 'mu_R', 'mu_T', 'target_div', 'pop_id'])

#assays_info_repl.to_csv('assays_info_repl.csv', index=False)
#populations_info_repl.to_csv('populations_info_repl.csv', index=False)